### Interactive Notebook UI – How to use this demo

This section provides a simple user interface for our **Music Mood Classifier** directly inside the notebook, so you can run and inspect the model without starting a separate Streamlit app.

**Manual feature entry (tab “Manual entry”)**

- Use the sliders to set the Spotify audio features for a single track  
  (e.g. danceability, energy, valence, tempo).  
- Click **“Predict genre”** to run the trained model on this configuration.  
- The notebook will display:
  - the **predicted genre** for that track  
  - the **top 5 predicted genres with probabilities**, to show model confidence.


This UI uses the **same trained model and preprocessing pipeline** as our Streamlit apps, so predictions here are identical to what you see in the web UI.


## Environment and dependencies

This notebook assumes you have already created the project environment from the repository root, for example:

```bash
pip install -r requirements.txt
```

Required packages for this UI include:

- `lightgbm`
- `ipywidgets`
- `pandas`, `numpy`, `scikit-learn`
- Local project package (for `src.feature_engineering.MusicFeatureEngineer`)

In [1]:
!pip install lightgbm==4.6.0
!pip install ipywidgets==8.1.2

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [24]:
"""
Imports and project paths.

This cell:
- configures the project root on `sys.path` so local modules can be imported
- imports standard libraries and third-party dependencies
"""

from __future__ import annotations

import sys
import warnings
from pathlib import Path
import pickle
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# ---------------------------------------------------------------------
# Project root and local imports
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import MusicFeatureEngineer  # noqa: F401  (needed for unpickling)

# Paths to persisted artefacts
MODEL_PATH = PROJECT_ROOT / "models" / "final_model.pkl"
PIPELINE_PATH = PROJECT_ROOT / "models" / "preprocessor.pkl"
ENCODER_PATH = PROJECT_ROOT / "models" / "label_encoder.pkl"

In [25]:
"""
Configuration for audio features and simple playlist suggestions.

FEATURE_CONFIG drives both the model input order and the widgets.
"""

# Audio feature metadata: (display name, min, max, default, step, help_text)
FEATURE_CONFIG: Dict[str, Tuple] = {
    "popularity":       ("Popularity",        0,   100,   50,     1,   "Spotify popularity score (0=obscure, 100=viral)"),
    "duration_ms":      ("Duration (ms)",     0,   600_000, 210_000, 1_000, "Track length in milliseconds"),
    "explicit":         ("Explicit",          0,   1,     0,      1,   "1 if explicit lyrics, 0 otherwise"),
    "danceability":     ("Danceability",      0.0, 1.0,   0.6,    0.01, "Suitability for dancing (0=low, 1=high)"),
    "energy":           ("Energy",            0.0, 1.0,   0.7,    0.01, "Perceptual intensity (0=calm, 1=intense)"),
    "key":              ("Key",               0,   11,    5,      1,   "Musical key (0=C, 1=C#, ..., 11=B)"),
    "loudness":         ("Loudness (dB)",    -60.0, 0.0, -7.0,    0.1, "Average loudness in decibels"),
    "mode":             ("Mode",              0,   1,     1,      1,   "1=major, 0=minor"),
    "speechiness":      ("Speechiness",      0.0, 1.0,   0.05,   0.01,"Presence of spoken words"),
    "acousticness":     ("Acousticness",     0.0, 1.0,   0.2,    0.01,"Confidence the track is acoustic"),
    "instrumentalness": ("Instrumentalness", 0.0, 1.0,   0.0,    0.01,"Probability of no vocals"),
    "liveness":         ("Liveness",         0.0, 1.0,   0.12,   0.01,"Probability of live recording"),
    "valence":          ("Valence",          0.0, 1.0,   0.5,    0.01,"Musical positiveness (0=sad, 1=happy)"),
    "tempo":            ("Tempo (BPM)",      0.0, 250.0, 120.0,  0.5, "Estimated beats per minute"),
    "time_signature":   ("Time Signature",   1,   7,     4,      1,   "Beats per bar"),
}

# Minimal playlist suggestion dictionary (subset of your Streamlit version)
import urllib.parse

PLAYLISTS = {
    "acoustic": [
        {"name": "Peaceful Piano", "desc": "Soft instrumentals for quiet moments", "query": "peaceful piano ambient playlist"},
        {"name": "Coffee Shop Acoustic", "desc": "Gentle singer-songwriter for slow mornings", "query": "coffee shop acoustic playlist"},
    ],
    "dance": [
        {"name": "Today's Top Hits", "desc": "Current high-energy pop hits", "query": "todays top hits pop playlist"},
        {"name": "Party Anthems", "desc": "Dance floor fillers", "query": "party anthems dance floor playlist"},
    ],
    "electronic": [
        {"name": "Electronic Focus", "desc": "Deep house and ambient for focus", "query": "electronic focus deep house playlist"},
    ],
    "heavy": [
        {"name": "Metal Classics", "desc": "Heavy rock and metal hall of fame", "query": "metal classics rock playlist"},
    ],
    "vocal": [
        {"name": "Hip-Hop Essentials", "desc": "Iconic hip-hop and rap tracks", "query": "hip hop rap essentials playlist"},
    ],
}

In [26]:
"""
Core utilities:
- loading trained artefacts
- single-record prediction
- batch prediction for DataFrames
"""

def load_artefacts():
    """
    Load trained model, preprocessing pipeline, and label encoder.

    Returns
    -------
    model :
        Fitted classifier.
    pipeline :
        Fitted preprocessing pipeline.
    label_encoder :
        Fitted LabelEncoder mapping class indices to genre labels.
    """
    with open(MODEL_PATH, "rb") as f:
        model = pickle.load(f)
    with open(PIPELINE_PATH, "rb") as f:
        pipeline = pickle.load(f)
    with open(ENCODER_PATH, "rb") as f:
        label_encoder = pickle.load(f)
    return model, pipeline, label_encoder


def predict_single(record: dict, model, pipeline, label_encoder):
    """
    Predict the genre of a single track.

    Parameters
    ----------
    record :
        Dictionary mapping feature name to numeric value.

    Returns
    -------
    genre : str
        Predicted genre label.
    proba_series : pandas.Series
        Class probabilities indexed by genre label, sorted descending.
    """
    df = pd.DataFrame([record])
    X_scaled = pipeline.transform(df)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        label_int = model.predict(X_scaled)[0]
        proba = model.predict_proba(X_scaled)[0]

    genre = label_encoder.inverse_transform([label_int])[0]
    proba_series = (
        pd.Series(proba, index=label_encoder.classes_)
        .sort_values(ascending=False)
    )
    return genre, proba_series


def predict_batch(df_raw: pd.DataFrame, model, pipeline, label_encoder) -> pd.DataFrame:
    """
    Run batch inference on a DataFrame of audio features.

    Any missing feature columns are added and filled with zeros.

    Parameters
    ----------
    df_raw :
        Input DataFrame with one row per track.

    Returns
    -------
    result_df :
        Copy of the input with additional columns:
        - ``predicted_genre`` : predicted label per row
        - ``confidence``      : highest class probability (in %)
    """
    feature_cols = list(FEATURE_CONFIG.keys())
    for col in feature_cols:
        if col not in df_raw.columns:
            df_raw[col] = 0

    X = df_raw[feature_cols].copy()
    X_scaled = pipeline.transform(X)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        y_pred = model.predict(X_scaled)
        proba = model.predict_proba(X_scaled)

    labels = label_encoder.inverse_transform(y_pred)
    result = df_raw.copy()
    result["predicted_genre"] = labels
    result["confidence"] = (proba.max(axis=1) * 100).round(1)
    return result

In [31]:
"""
Load the full Spotify dataset for dynamic song recommendations.

We keep only the columns needed for filtering and for building Spotify links.
"""

DATA_PATH = PROJECT_ROOT / "dataset.csv"  # adjust if your file lives elsewhere

full_df = pd.read_csv(DATA_PATH)

# Columns needed for recommendations
REC_COLUMNS = ["track_name", "artists", "track_genre", "track_id", "popularity"]
available_cols = [c for c in REC_COLUMNS if c in full_df.columns]
rec_df = full_df[available_cols].copy()

In [32]:
def build_track_url(track_id: str) -> str:
    """
    Build a Spotify track URL from a track_id.
    """
    return f"https://open.spotify.com/track/{track_id}"


def get_top_tracks_for_genre(genre: str, k: int = 5) -> pd.DataFrame:
    """
    Select up to k recommended tracks from the dataset for a given genre.

    Strategy
    --------
    - Filter tracks by the predicted `track_genre`
    - Sort by popularity descending (if that column is available)
    - Return the top k rows with Spotify URLs.

    Parameters
    ----------
    genre :
        Predicted genre label.
    k :
        Maximum number of recommended tracks.

    Returns
    -------
    pandas.DataFrame
        Columns: track_name, artists, spotify_url.
    """
    if "track_genre" not in rec_df.columns:
        return pd.DataFrame()

    subset = rec_df[rec_df["track_genre"] == genre].copy()
    if subset.empty:
        return subset

    if "popularity" in subset.columns:
        subset = subset.sort_values("popularity", ascending=False)

    subset = subset.head(k)

    if "track_id" in subset.columns:
        subset["spotify_url"] = subset["track_id"].apply(build_track_url)
    else:
        subset["spotify_url"] = ""

    return subset[["track_name", "artists", "spotify_url"]]

In [27]:
"""
Load trained artefacts once for the whole UI.

If anything fails here, fix the paths or run the training pipeline again.
"""

model, pipeline, le = load_artefacts()
print("Loaded model, pipeline, and label encoder successfully.")

Loaded model, pipeline, and label encoder successfully.


C:\ProgramData\Anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\ProgramData\Anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\ProgramData\Anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.6.1 when using version 1.5.1. This might lead to breaking code or

In [63]:
"""
Widget builders for the interactive UI:
- manual feature entry tab
- batch CSV prediction tab
"""

def build_manual_ui():
    """
    Create the manual feature-entry UI with sliders and a predict button.

    Returns
    -------
    ipywidgets.VBox
        Container widget that can be placed in a tab.
    """
    sliders: Dict[str, widgets.Widget] = {}

    # Create a slider for each feature
    for key, (label, lo, hi, default, step, help_text) in FEATURE_CONFIG.items():
        if isinstance(step, int):
            w = widgets.IntSlider(
                description=label,
                min=int(lo),
                max=int(hi),
                value=int(default),
                step=int(step),
                continuous_update=False,
            )
        else:
            w = widgets.FloatSlider(
                description=label,
                min=float(lo),
                max=float(hi),
                value=float(default),
                step=float(step),
                continuous_update=False,
            )
        w.style.description_width = "160px"
        sliders[key] = w

    button_predict = widgets.Button(
        description="Predict genre",
        button_style="success",
        icon="music",
    )
    output_single = widgets.Output()

    def on_predict_click(_):
        """Callback for the 'Predict genre' button."""
        with output_single:
            output_single.clear_output()
            record = {k: w.value for k, w in sliders.items()}
            genre, proba_series = predict_single(record, model, pipeline, le)

            # --- Styled prediction summary card ---
            top5 = proba_series.head(5).to_frame("probability")
            top5["probability"] = (top5["probability"] * 100).round(1)
    
            summary_html = f"""
            <style>
            .pred-card {{
                margin-top: 8px;
                margin-bottom: 8px;
                padding: 14px 18px;
                border-radius: 14px;
                background: linear-gradient(135deg, #111827, #020617);
                color: #e5e7eb;
                font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
                box-shadow: 0 4px 12px rgba(0,0,0,0.3);
            }}
            .pred-title {{
                font-size: 0.95rem;
                letter-spacing: 0.08em;
                text-transform: uppercase;
                color: #9ca3af;
                margin-bottom: 4px;
            }}
            .pred-genre {{
                font-size: 1.4rem;
                font-weight: 700;
                letter-spacing: 0.02em;
                margin-bottom: 10px;
            }}
            .prob-table {{
                width: 100%;
                border-collapse: collapse;
                font-size: 0.85rem;
            }}
            .prob-table th,
            .prob-table td {{
                padding: 4px 6px;
            }}
            .prob-table th {{
                text-align: left;
                color: #9ca3af;
                font-weight: 500;
            }}
            .prob-table tr:nth-child(odd) td {{
                background: rgba(15,23,42,0.7);
            }}
            .prob-table tr:nth-child(even) td {{
                background: rgba(15,23,42,0.4);
            }}
            .prob-bar {{
                height: 4px;
                border-radius: 999px;
                background: rgba(30,64,175,0.6);
                overflow: hidden;
            }}
            .prob-bar-fill {{
                height: 100%;
                border-radius: 999px;
                background: linear-gradient(90deg, #38bdf8, #6366f1);
            }}
            </style>
            <div class="pred-card">
              <div class="pred-title">Predicted genre</div>
              <div class="pred-genre">{genre}</div>
              <table class="prob-table">
                <thead>
                  <tr><th>Genre</th><th>Confidence</th></tr>
                </thead>
                <tbody>
            """
    
            for g, p in top5["probability"].items():
                summary_html += f"""
                  <tr>
                    <td>{g}</td>
                    <td>
                      {p:.1f}% 
                      <div class="prob-bar">
                        <div class="prob-bar-fill" style="width:{p:.1f}%"></div>
                      </div>
                    </td>
                  </tr>
                """
    
            summary_html += """
                </tbody>
              </table>
            </div>
            """
    
            display(widgets.HTML(summary_html))
    
            # --- Compact playlist section under the card ---
            if genre in PLAYLISTS:
                pl_html = "<b>Suggested playlists</b><br>"
                for pl in PLAYLISTS[genre]:
                    url = "https://open.spotify.com/search/" + urllib.parse.quote(pl["query"])
                    pl_html += f"{pl['name']} – {pl['desc']}<br><a href='{url}' target='_blank'>{url}</a><br>"
                display(widgets.HTML(pl_html))
            # --- Dynamic top‑5 song recommendations from the dataset ---
            top_tracks = get_top_tracks_for_genre(genre, k=5)
            if not top_tracks.empty:
                cards_html = """
                <style>
                .rec-card-container {
                    display: flex;
                    flex-direction: column;
                    gap: 10px;
                    margin-top: 8px;
                }
                .rec-card {
                    display: flex;
                    align-items: center;
                    justify-content: space-between;
                    padding: 10px 14px;
                    border-radius: 14px;
                    background: linear-gradient(135deg, #111827, #1f2937);
                    box-shadow: 0 4px 14px rgba(0,0,0,0.35);
                    color: #e5e7eb;
                    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
                }
                .rec-main {
                    display: flex;
                    align-items: center;
                    gap: 10px;
                }
                .rec-text-title {
                    font-size: 0.98rem;
                    font-weight: 600;
                    letter-spacing: 0.01em;
                }
                .rec-text-artist {
                    font-size: 0.78rem;
                    color: #9ca3af;
                }
                .rec-play-btn {
                    width: 34px;
                    height: 34px;
                    border-radius: 999px;
                    border: none;
                    background: radial-gradient(circle at 30% 20%, #38bdf8, #6366f1);
                    display: flex;
                    align-items: center;
                    justify-content: center;
                    box-shadow: 0 0 0 2px rgba(15,23,42,0.7);
                    cursor: pointer;
                }
                .rec-play-btn span {
                    margin-left: 2px;
                    color: white;
                    font-size: 0.9rem;
                }
                </style>
                <h4>Top 5 recommended songs</h4>
                <div class="rec-card-container">
                """
    
                # Build one HTML card per track
                for _, row in top_tracks.iterrows():
                    title = row["track_name"]
                    artist = row["artists"]
                    url = row["spotify_url"]
                    cards_html += f"""
                    <div class="rec-card">
                        <div class="rec-main">
                            <div>
                                <div class="rec-text-title">{title}</div>
                                <div class="rec-text-artist">{artist}</div>
                            </div>
                        </div>
                        <a class="rec-play-btn" href="{url}" target="_blank">
                            <span>▶</span>
                        </a>
                    </div>
                    """
    
                cards_html += "</div>"
    
                display(widgets.HTML(cards_html))
    button_predict.on_click(on_predict_click)

    grid = widgets.GridBox(
        list(sliders.values()),
        layout=widgets.Layout(
            grid_template_columns="repeat(3, 300px)",
            grid_gap="8px 16px",
        ),
    )

    return widgets.VBox(
        [
            widgets.HTML("<h3>Manual feature entry</h3>"),
            grid,
            button_predict,
            output_single,
        ]
    )

In [64]:
"""
Interactive UI section.

This cell builds the two tabs and displays them:
- 'Manual entry' for a single track
"""

tab_manual = build_manual_ui()

ui_tabs = widgets.Tab(children=[tab_manual])
ui_tabs.set_title(0, "Manual entry")

display(ui_tabs)